# Librerías

In [18]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

# Setup

In [9]:
datos = pd.read_csv('train.csv')
df = pd.DataFrame(datos)
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\javie\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


# Análisis exploratorio

In [4]:
df.head(10)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


In [10]:
# Convertir todo a minúsculas
df["texto_normalizado"] = df["text"].str.lower()
#   Eliminar URL
df["texto_normalizado"] = df["texto_normalizado"].str.replace(
    r"http\S+|www\S+",
    "",
    regex=True
)
# Eliminar menciones
df["texto_normalizado"] = df["texto_normalizado"].str.replace(
    r"@\w+",
    "",
    regex=True
)
# Eliminar hashtags
df["texto_normalizado"] = df["texto_normalizado"].str.replace(
    "#+",
    "",
    regex=False
)
# Eliminar signos de puntuacion
df["texto_normalizado"] = df["texto_normalizado"].str.replace(
    r"[^\w\s]",
    " ",
    regex=True
)
# eliminar stopwords
df["texto_normalizado"] = df["texto_normalizado"].apply(
    lambda texto: " ".join(
        palabra
        for palabra in texto.split()
        if palabra not in stop_words
    )
)

In [15]:
# Frecuencia de palabras
tweets_desastre = df[
    df["target"] == 1
]["texto_normalizado"]

tweets_no_desastre = df[
    df["target"] == 0
]["texto_normalizado"]

palabras_desastre = Counter(
    " ".join(tweets_desastre).split()
)

palabras_no_desastre = Counter(
    " ".join(tweets_no_desastre).split()
)

print("Palabras más comunes en tweets de desastre:")
print(palabras_desastre.most_common(20))
print("------------------------------")
print("Palabras más comunes en tweets no de desastre:")
print(palabras_no_desastre.most_common(20))

Palabras más comunes en tweets de desastre:
[('fire', 182), ('û_', 172), ('news', 139), ('amp', 135), ('disaster', 121), ('via', 121), ('california', 115), ('suicide', 112), ('police', 109), ('people', 105), ('2', 102), ('killed', 95), ('like', 94), ('hiroshima', 92), ('storm', 89), ('fires', 86), ('crash', 85), ('families', 81), ('train', 79), ('emergency', 77)]
------------------------------
Palabras más comunes en tweets no de desastre:
[('like', 254), ('amp', 209), ('û_', 171), ('new', 170), ('get', 163), ('one', 131), ('body', 116), ('2', 112), ('would', 101), ('via', 99), ('video', 96), ('people', 94), ('love', 90), ('day', 86), ('know', 86), ('back', 85), ('time', 84), ('got', 84), ('full', 84), ('3', 82)]


In [16]:
# Palabras que aparecen en tweets de desastre y no desastre
comunes = (
    set(palabras_desastre.keys()) &
    set(palabras_no_desastre.keys())
)

frecuencias_comunes = []

for palabra in comunes:
    frecuencias_comunes.append({
        "palabra": palabra,
        "desastre": palabras_desastre[palabra],
        "no_desastre": palabras_no_desastre[palabra]
    })

df_comunes = pd.DataFrame(frecuencias_comunes)

df_comunes["total"] = (
    df_comunes["desastre"] +
    df_comunes["no_desastre"]
)
df_comunes.sort_values(
    "total",
    ascending=False
).head(20)

,palabra,desastre,no_desastre,total
530,like,94,254,348
1839,amp,135,209,344
647,û_,172,171,343
832,fire,182,72,254
150,get,66,163,229
2885,new,56,170,226
2961,via,121,99,220
1997,2,102,112,214
3845,people,105,94,199
1506,news,139,57,196


In [17]:
# Proporción de aparición en tweets de desastre y no desastre
df_comunes["prop_desastre"] = (
    df_comunes["desastre"] /
    df_comunes["total"]
)

df_comunes["prop_no_desastre"] = (
    df_comunes["no_desastre"] /
    df_comunes["total"]
)

In [19]:
# Biogramas
# Para desaste
vectorizador_bi = CountVectorizer(
    ngram_range=(2, 2)
)

X_bi_desastre = vectorizador_bi.fit_transform(
    tweets_desastre
)

frecuencias = X_bi_desastre.sum(axis=0).A1

bigramas_desastre = pd.DataFrame({
    "bigrama": vectorizador_bi.get_feature_names_out(),
    "frecuencia": frecuencias
})

bigramas_desastre.sort_values(
    "frecuencia",
    ascending=False
).head(20)

# Para no desastre
vectorizador_bi_no = CountVectorizer(
    ngram_range=(2, 2)
)

X_bi_no = vectorizador_bi_no.fit_transform(
    tweets_no_desastre
)

frecuencias_no = X_bi_no.sum(axis=0).A1

bigramas_no_desastre = pd.DataFrame({
    "bigrama": vectorizador_bi_no.get_feature_names_out(),
    "frecuencia": frecuencias_no
})

bigramas_no_desastre.sort_values(
    "frecuencia",
    ascending=False
).head(20)

,bigrama,frecuencia
5686,cross body,39
14020,liked video,34
10850,gt gt,30
9697,full read,28
3271,body bag,27
9704,full û_,25
3273,body bagging,24
3802,burning buildings,23
3274,body bags,22
14400,looks like,21
